In [ ]:
# Example 1: async builtins -- list(), enumerate(), map() over async iterables.
import asyncstdlib as a

async def gen():
    for i in range(5):
        yield i * 10

collected = await a.list(gen())
print(f"a.list(gen()) = {collected}")
assert collected == [0, 10, 20, 30, 40]

enumerated = await a.list(a.enumerate(gen()))
print(f"a.enumerate = {enumerated}")
assert enumerated == [(0, 0), (1, 10), (2, 20), (3, 30), (4, 40)]

mapped = await a.list(a.map(lambda x: x + 1, gen()))
print(f"a.map(+1) = {mapped}")
assert mapped == [1, 11, 21, 31, 41]


In [ ]:
# Example 2: async itertools -- chain, islice, takewhile, accumulate.
import asyncstdlib as a

async def src(values):
    for v in values:
        yield v

chained = await a.list(a.itertools.chain(src([1, 2, 3]), src([4, 5])))
print(f"chain = {chained}")
assert chained == [1, 2, 3, 4, 5]

sliced = await a.list(a.itertools.islice(src(range(100)), 3, 7))
print(f"islice(3,7) = {sliced}")
assert sliced == [3, 4, 5, 6]

took = await a.list(a.itertools.takewhile(lambda x: x < 4, src([1, 2, 3, 4, 5, 1])))
print(f"takewhile<4 = {took}")
assert took == [1, 2, 3]


In [ ]:
# Example 3: async functools -- reduce + lru_cache on an async function.
import asyncstdlib as a

async def src():
    for v in [1, 2, 3, 4, 5]:
        yield v

total = await a.functools.reduce(lambda x, y: x + y, src(), 0)
print(f"reduce sum = {total}")
assert total == 15

call_count = 0

@a.functools.lru_cache(maxsize=8)
async def fetch(key):
    global call_count
    call_count += 1
    return key * 2

assert await fetch(5) == 10
assert await fetch(5) == 10
assert await fetch(6) == 12
print(f"call_count after 3 calls (5,5,6) = {call_count}")
assert call_count == 2


In [ ]:
# Example 4: async context managers -- ExitStack-equivalent + scoped_iter.
import asyncstdlib as a

events = []

@a.contextmanager
async def named(label):
    events.append(f"enter:{label}")
    try:
        yield label
    finally:
        events.append(f"exit:{label}")

async with named("outer") as o, named("inner") as i:
    events.append(f"body:{o}/{i}")

print(events)
assert events == ["enter:outer", "enter:inner", "body:outer/inner", "exit:inner", "exit:outer"]


In [ ]:
# Example 5: zip / tee / groupby on async iterables.
import asyncstdlib as a

async def src(values):
    for v in values:
        yield v

zipped = await a.list(a.zip(src([1, 2, 3]), src(["a", "b", "c"])))
print(f"zip = {zipped}")
assert zipped == [(1, "a"), (2, "b"), (3, "c")]

async with a.itertools.tee(src([10, 20, 30]), n=2) as (t1, t2):
    out1 = await a.list(t1)
    out2 = await a.list(t2)
print(f"tee outputs = {out1} / {out2}")
assert out1 == out2 == [10, 20, 30]

# groupby is an async iterator (not an async context manager) — iterate it directly.
groups = []
async for key, grp in a.itertools.groupby(src([1, 1, 2, 2, 2, 3]), key=lambda x: x):
    groups.append((key, await a.list(grp)))
print(f"groupby = {groups}")
assert groups == [(1, [1, 1]), (2, [2, 2, 2]), (3, [3])]
